In [36]:
from pathlib import Path
import pandas as pd


In [37]:
excel_path = next(Path("D:/hcdcarbody/simulation/data").glob("章丘车身库数据v20260311.xlsx"))
pd1 = pd.read_excel(excel_path, sheet_name="出库计划")
pd2 = pd.read_excel(excel_path, sheet_name="任务")

In [38]:
pd_data = pd1[pd1["来源"] == "Excel导入"].copy()

In [39]:
# add outlet mapping column directly to pd_data
mes_type_col = pd_data.columns[4]  # mes plan type
outlet_map = {
    'WB1': 'L1C17',
    'WB2': 'L2C1',
    'PB1': 'L1C1',
    'PB3': 'L3C17',
}
pd_data['out_line'] = pd_data[mes_type_col].map(outlet_map).fillna('UNMAPPED')
pd_data.head()


,计划序号,mes计划编号,所属层,类型,mes计划类型,车身类型,状态,来源,车身类型.1,计划数量,...,完成时间,滑橇类型,滑橇号,滑橇状态,车身编号,颜色,车身配置,mes生产序号,process_state_number,out_line
0,210514,202412280434,1,白车身出库,WB1,卡车,已完成,Excel导入,白车身,1,...,2025-01-02 08:41:11.743,0,NaN,1.0,PLK001416,W1,CS37；TGA/G7/L-15超平顶/奔驰白/无后窗/钢杠/带中涂/独立暖风/出口车；WG...,434,SS24100562/0001-0001,L1C17
1,210515,202412280435,1,白车身出库,WB1,卡车,已完成,Excel导入,白车身,1,...,2025-01-02 08:45:09.463,0,NaN,1.0,PLK001417,W1,CS37；TGA/G7/L-15超平顶/奔驰白/无后窗/钢杠/带中涂/独立暖风/出口车；WG...,435,SS24100562/0001-0001,L1C17
2,210516,202412280436,1,白车身出库,WB1,卡车,已完成,Excel导入,白车身,1,...,2025-01-02 08:48:18.410,0,NaN,1.0,PLK001418,W1,CS37；TGA/G7/L-15超平顶/奔驰白/无后窗/钢杠/带中涂/独立暖风/出口车；WG...,436,SS24100562/0001-0001,L1C17
3,210517,202412280437,1,白车身出库,WB1,卡车,已完成,Excel导入,白车身,1,...,2025-01-02 08:50:31.597,0,NaN,1.0,PLK001419,W1,CS37；TGA/G7/L-15超平顶/奔驰白/无后窗/钢杠/带中涂/独立暖风/出口车；WG...,437,SS24100562/0001-0001,L1C17
4,210623,202412010290,2,白车身出库,WB2,卡车,已完成,Excel导入,白车身,1,...,2025-01-02 07:56:42.680,0,NaN,1.0,PLT003045,W1,CS22；TGA/TX/L-32/咖啡金/带后窗/钢杠/带中涂/机械油门过线孔/C5车门/仅...,290,SS24110610/0030-0031,L2C1


In [40]:
pd_data2 = pd2[(pd2["类型"].isin(["入库","白车身出库","彩车身出库"]))].copy()

In [41]:
# remove outbound tasks in pd_data2 that belong to WMS-source plans in pd1 (robust)
source_col_pd1 = next((c for c in pd1.columns if "来源" in str(c)), pd1.columns[7])
type_col_pd1 = next((c for c in pd1.columns if "类型" in str(c)), pd1.columns[3])
plan_col_pd1 = next((c for c in pd1.columns if "计划序号" in str(c)), pd1.columns[0])

type_col_pd2 = next((c for c in pd_data2.columns if "类型" in str(c)), pd_data2.columns[1])
plan_col_pd2 = next((c for c in pd_data2.columns if "计划序号" in str(c)), pd_data2.columns[4])

# pd1: WMS + 出库 的计划序号（数值化）
wms_out_plan_ids = set(
    pd.to_numeric(
        pd1[
            pd1[source_col_pd1].astype(str).str.contains("WMS", na=False)
            & pd1[type_col_pd1].astype(str).str.contains("出库", na=False)
        ][plan_col_pd1],
        errors="coerce",
    ).dropna().astype("Int64")
)

before_cnt = len(pd_data2)

type_text_pd2 = pd_data2[type_col_pd2].astype(str)
# 更稳：只要不是“入库”就视为出库侧事件
is_outbound_pd2 = ~type_text_pd2.str.contains("入库", na=False)

plan_num_pd2 = pd.to_numeric(pd_data2[plan_col_pd2], errors="coerce").astype("Int64")
hit_mask = is_outbound_pd2 & plan_num_pd2.isin(wms_out_plan_ids)

pd_data2 = pd_data2[~hit_mask].copy()
after_cnt = len(pd_data2)

print(f"wms_out_plan_ids: {len(wms_out_plan_ids)}")
print(f"pd_data2 matched outbound rows to drop: {hit_mask.sum()}")
print(f"pd_data2 remove WMS-outbound by plan_id: {before_cnt} -> {after_cnt}")


wms_out_plan_ids: 1750
pd_data2 matched outbound rows to drop: 3788
pd_data2 remove WMS-outbound by plan_id: 249556 -> 245768


In [43]:
# 回填 out_line 并同时做 in/out 严格配对过滤（只保留真正使用到的记录）
from collections import defaultdict, deque, Counter
import pandas as pd


def norm_id(x):
    return str(x).strip().upper().replace('-', '').replace('_', '').replace(' ', '')


# ===== 1) 列名定位 =====
out_plan_col = next((c for c in pd_data.columns if '计划序号' in str(c)), pd_data.columns[0])
out_car_body_col = next((c for c in pd_data.columns if '车身编号' in str(c)), pd_data.columns[17])
out_skid_state_col = next((c for c in pd_data.columns if '滑橇状态' in str(c)), pd_data.columns[16])
out_line_col = next((c for c in pd_data.columns if 'out_line' in str(c).lower()), pd_data.columns[-1])
out_start_col = next((c for c in pd_data.columns if '开始时间' in str(c)), pd_data.columns[12])

type_col = next((c for c in pd_data2.columns if '类型' in str(c)), pd_data2.columns[1])
plan_col = next((c for c in pd_data2.columns if '计划序号' in str(c)), pd_data2.columns[4])
rfid_col = next((c for c in pd_data2.columns if str(c).lower() == 'rfid'), pd_data2.columns[12])
start_col = next((c for c in pd_data2.columns if '开始时间' in str(c)), pd_data2.columns[10])

blocked = {'000000000000000', '100000000000000'}
blocked_norm = {norm_id(x) for x in blocked}

# ===== 2) 标准化数据 =====
before_out_all = len(pd_data)
before_in_all = len(pd_data2)

pd1 = pd_data.copy().reset_index().rename(columns={'index': '_orig_idx'})
pd1['_plan_num'] = pd.to_numeric(pd1[out_plan_col], errors='coerce').astype('Int64')
pd1['_sku10'] = (
    pd1[out_skid_state_col].astype(str).str.extract(r'(\d)', expand=False).fillna('0')
    + pd1[out_car_body_col].map(norm_id).str[-9:]
)
pd1['_ts'] = pd.to_datetime(pd1[out_start_col], errors='coerce')
pd1 = pd1.sort_values(['_ts', out_plan_col, '_orig_idx']).reset_index(drop=True)

work = pd_data2.copy()
work['_row_id'] = work.index
work['_rfid_full'] = work[rfid_col].map(norm_id)
work['_sku10'] = work['_rfid_full'].str[-10:]
work['_plan_num'] = pd.to_numeric(work[plan_col], errors='coerce').astype('Int64')
work['_ts'] = pd.to_datetime(work[start_col], errors='coerce')
work = work[~work['_rfid_full'].isin(blocked_norm)].copy()

in2 = work[work[type_col].astype(str).str.contains('入库', na=False)].copy()
out2 = work[work[type_col].astype(str).str.contains('出库', na=False)].copy()
out2 = out2.sort_values(['_ts', '_row_id']).reset_index(drop=True)

# ===== 3) 先把 pd_data 出库按 (plan, sku10) 配额匹配到 pd_data2 出库 =====
buckets = defaultdict(deque)
for _, r in pd1.iterrows():
    k = (r['_plan_num'], r['_sku10'])
    if pd.notna(r['_plan_num']) and isinstance(r['_sku10'], str) and r['_sku10']:
        buckets[k].append(int(r['_orig_idx']))

out2_to_pd1idx = {}
for _, r in out2.iterrows():
    k = (r['_plan_num'], r['_sku10'])
    if buckets[k]:
        out2_to_pd1idx[int(r['_row_id'])] = buckets[k].popleft()

matched_out2_ids_pre = set(out2_to_pd1idx.keys())

# ===== 4) 在 pd_data2 内按 full RFID + 时间做 in->out FIFO（仅匹配到 pd_data 的出库参与） =====
out2_m = out2[out2['_row_id'].isin(matched_out2_ids_pre)].copy()
in2['_event'] = 'in'
out2_m['_event'] = 'out'

events = pd.concat([
    in2[['_rfid_full', '_ts', '_row_id', '_event']],
    out2_m[['_rfid_full', '_ts', '_row_id', '_event']],
], ignore_index=True)
events['_ord'] = events['_event'].map({'in': 0, 'out': 1})
events = events.sort_values(['_ts', '_ord', '_row_id']).reset_index(drop=True)

pending_in = defaultdict(deque)
pairs = []  # (in_row_id, out_row_id)
dropped_out = 0
for _, r in events.iterrows():
    key = r['_rfid_full']
    rid = int(r['_row_id'])
    if r['_event'] == 'in':
        pending_in[key].append(rid)
    else:
        if pending_in[key]:
            pairs.append((pending_in[key].popleft(), rid))
        else:
            dropped_out += 1

extra_inbound = sum(len(q) for q in pending_in.values())

use_in_ids = {i for i, _ in pairs}
use_out2_ids = {o for _, o in pairs}

# ===== 5) 基于 pair 结果同步裁剪 pd_data / pd_data2，并回填入库 out_line =====
use_pd1_idxs = [out2_to_pd1idx[o] for o in use_out2_ids if o in out2_to_pd1idx]
pd_data = pd_data.loc[sorted(set(use_pd1_idxs))].copy()

# 入库回填 out_line：in_row -> 对应 out_row -> pd_data out_line
out2_to_outline = {
    o: pd_data.loc[out2_to_pd1idx[o], out_line_col]
    for o in use_out2_ids
    if o in out2_to_pd1idx and out2_to_pd1idx[o] in pd_data.index
}
in_to_outline = {i: out2_to_outline[o] for i, o in pairs if o in out2_to_outline}

pd_data2 = in2[in2['_row_id'].isin(use_in_ids)].copy()
pd_data2['out_line'] = pd_data2['_row_id'].map(in_to_outline)

# 清理临时列
pd_data2.drop(columns=['_row_id', '_rfid_full', '_sku10', '_plan_num', '_ts', '_event'], errors='ignore', inplace=True)

print(f'pd_data(filtered outbound): {before_out_all} -> {len(pd_data)}')
print(f'pd_data2(filtered inbound): {before_in_all} -> {len(pd_data2)}')
print(f'dropped outbound without prior inbound: {dropped_out}')
print(f'dropped extra inbound without outbound: {extra_inbound}')
print(f'inbound out_line filled: {int(pd_data2["out_line"].notna().sum())}/{len(pd_data2)}')
print(f'in/out balanced? {len(pd_data2) == len(pd_data)}')

pd_data.head(), pd_data2.head()


pd_data(filtered outbound): 109541 -> 105192
pd_data2(filtered inbound): 245768 -> 105192
dropped outbound without prior inbound: 4349
dropped extra inbound without outbound: 12361
inbound out_line filled: 105192/105192
in/out balanced? True


(       计划序号       mes计划编号  所属层     类型 mes计划类型 车身类型   状态       来源 车身类型.1  计划数量  \
 120  210786  202412280490    1  白车身出库     WB1   卡车  已完成  Excel导入    白车身     1   
 121  210787  202412280491    1  白车身出库     WB1   卡车  已完成  Excel导入    白车身     1   
 122  210788  202412280492    1  白车身出库     WB1   卡车  已完成  Excel导入    白车身     1   
 123  210789  202412280493    1  白车身出库     WB1   卡车  已完成  Excel导入    白车身     1   
 124  210790  202412280494    1  白车身出库     WB1   卡车  已完成  Excel导入    白车身     1   
 
      ...                    完成时间 滑橇类型  滑橇号 滑橇状态       车身编号  颜色  \
 120  ... 2025-01-02 11:10:19.230    0  NaN  1.0  RAT000001  W1   
 121  ... 2025-01-02 11:12:04.830    0  NaN  1.0  RAT000002  W1   
 122  ... 2025-01-02 11:15:23.197    0  NaN  1.0  RAT000003  W1   
 123  ... 2025-01-02 11:18:27.953    0  NaN  1.0  RAT000004  W1   
 124  ... 2025-01-02 11:20:54.527    0  NaN  1.0  RAT000005  W1   
 
                                                   车身配置 mes生产序号  \
 120           CS26；TGA/G7H/L-47/月光

In [44]:
# check inbound out_line fill rate
type_col = next((c for c in pd_data2.columns if '类型' in str(c)), pd_data2.columns[1])
inbound_mask = pd_data2[type_col].astype(str).str.contains('入库', na=False)
total_in = int(inbound_mask.sum())
filled_in = int(pd_data2.loc[inbound_mask, 'out_line'].notna().sum()) if 'out_line' in pd_data2.columns else 0
print(f'inbound out_line filled: {filled_in}/{total_in} ({(filled_in/total_in*100) if total_in else 0:.2f}%)')


inbound out_line filled: 105192/105192 (100.00%)


In [45]:
# check each sku follows in-out-in-out order
car_body_col = next((c for c in pd_data.columns if "车身编号" in str(c)), pd_data.columns[17])
skid_state_col = next((c for c in pd_data.columns if "滑橇状态" in str(c)), pd_data.columns[16])
out_start_col = next((c for c in pd_data.columns if "开始时间" in str(c)), pd_data.columns[12])
rfid_col = next((c for c in pd_data2.columns if str(c).lower() == "rfid"), pd_data2.columns[13])
in_start_col = next((c for c in pd_data2.columns if "开始时间" in str(c)), pd_data2.columns[10])

blocked_rfid = {"000000000000000", "100000000000000"}

out_chk = pd_data.copy()
out_chk["sku10"] = out_chk[skid_state_col].astype(str).str.extract(r"(\d)", expand=False).fillna("0") + out_chk[car_body_col].astype(str).str.strip().str[-9:]
out_chk["event"] = "out"
out_chk["ts"] = pd.to_datetime(out_chk[out_start_col], errors="coerce")

in_chk = pd_data2.copy()
in_chk["rfid_raw"] = in_chk[rfid_col].astype(str).str.strip()
in_chk = in_chk[~in_chk["rfid_raw"].isin(blocked_rfid)].copy()
in_chk["sku10"] = in_chk["rfid_raw"].str[-10:]
in_chk["event"] = "in"
in_chk["ts"] = pd.to_datetime(in_chk[in_start_col], errors="coerce")

ev = pd.concat([in_chk[["sku10", "event", "ts"]], out_chk[["sku10", "event", "ts"]]], ignore_index=True)
ev = ev.sort_values(["sku10", "ts", "event"]).reset_index(drop=True)

violations = []
ok_count = 0
for sku, g in ev.groupby("sku10", sort=True):
    seq = g["event"].tolist()
    # Must start with in, then strictly alternate
    valid = True
    if not seq or seq[0] != "in":
        valid = False
    else:
        for i in range(1, len(seq)):
            if seq[i] == seq[i - 1]:
                valid = False
                break
    if valid:
        ok_count += 1
    else:
        violations.append({"sku10": sku, "len": len(seq), "sequence": "-".join(seq[:12])})

viol_df = pd.DataFrame(violations)
print(f"total sku: {ev['sku10'].nunique()}")
print(f"valid sku: {ok_count}")
print(f"invalid sku: {len(violations)}")
viol_df.head(30)


total sku: 99720
valid sku: 99702
invalid sku: 18


,sku10,len,sequence
0,1NFR000016,24,in-out-in-out-in-out-in-out-in-out-in-out
1,1NFR000017,24,in-out-in-out-in-out-in-out-in-out-in-out
2,1RJR000490,4,in-in-out-out
3,1rck000003,1,out
4,1rck000461,1,out
5,1rek000162,1,out
6,1rek000392,1,out
7,1rhk000485,1,out
8,1rhk000486,1,out
9,1rik000401,1,out


In [47]:
pd_data2.head()

,任务号,类型,所属层,状态,计划序号,源巷道号,源货位,目标巷道,目标货位,创建时间,开始时间,完成时间,rfid,out_line
11,440143,入库,1,完成,NaN,NaN,NaN,3号巷道,6-3-3,2025-01-02 08:37:15.877,2025-01-02 08:37:16.933,2025-01-02 08:39:18.010,002401241200239,L1C17
63,440193,入库,1,完成,NaN,NaN,NaN,3号巷道,6-1-2,2025-01-02 09:20:47.523,2025-01-02 09:20:48.617,2025-01-02 09:22:46.940,001542PLT002936,L3C17
71,440202,入库,1,完成,NaN,NaN,NaN,4号巷道,7-9-1,2025-01-02 09:28:13.893,2025-01-02 09:28:15.210,2025-01-02 09:29:17.087,001711RAT000005,L1C17
74,440204,入库,1,完成,NaN,NaN,NaN,1号巷道,2-5-3,2025-01-02 09:29:18.977,2025-01-02 09:29:20.063,2025-01-02 09:31:09.130,004491RAT000003,L1C17
76,440207,入库,1,完成,NaN,NaN,NaN,3号巷道,5-12-3,2025-01-02 09:30:58.847,2025-01-02 09:30:59.977,2025-01-02 09:32:42.817,001951RAT000001,L1C17


In [48]:
pd_data.to_excel("D:/hcdcarbody/simulation/data/outbound_task.xlsx", index=False)
pd_data2.to_excel("D:/hcdcarbody/simulation/data/inbound_task.xlsx", index=False)